# ATML PA3 Task1 — Colab prep + post only

**No training.** Uses checkpoints already on Drive.

### GPU frugality
1. Run **cells 1–2 on CPU runtime** (mount + verify + pip).
2. Then **Runtime → Change runtime type → GPU** (T4 is enough).
3. Run **cell 3** (GPU gate). Stop if it fails — do not continue.
4. Run **cell 4** (tiny smoke). Stop if it fails.
5. Run **cell 5** (`pipeline_prep`) once.
6. Run **cell 6** (`pipeline_post`) once.


In [ ]:
# Cell 1 — CPU OK: mount Drive + verify code/checkpoints (NO heavy compute)
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

TASK1 = Path('/content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA3/task1')
CKPT_SRC = TASK1 / 'colab_data' / 'checkpoints'
ART = TASK1 / 'results_colab'
ART.mkdir(parents=True, exist_ok=True)

assert (TASK1 / 'src' / 'run_stage.py').is_file(), TASK1
assert (TASK1 / 'src' / 'eval' / 'run_eval.py').read_text(encoding='utf-8').find('_patch_tokenizer_disable_thinking') >= 0
need = ['full_ft', 'full_ft_l2sp', 'lora_r4', 'lora_r16', 'oplora']
missing = [v for v in need if not (CKPT_SRC / v).is_dir()]
assert not missing, f'missing checkpoints on Drive: {missing}'
for v in need:
    steps = sorted(p.name for p in (CKPT_SRC / v).glob('step_*'))
    print(v, steps)
print('TASK1', TASK1)
print('ART', ART)


In [ ]:
# Cell 2 — CPU OK: install deps once (before switching to GPU is fine)
import subprocess, sys
pkgs = [
    'datasets', 'peft==0.14.0', 'accelerate', 'sentencepiece', 'pyyaml',
    'lm-eval', 'einops', 'safetensors', 'matplotlib', 'seaborn', 'pandas', 'tabulate',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers>=4.51.0', *pkgs])
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
print('deps ok')


In [ ]:
# Cell 3 — SWITCH TO GPU RUNTIME FIRST, then run this gate
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — Runtime→Change runtime type→GPU T4, then re-run this cell'
print(torch.cuda.get_device_name(0), 'VRAM_GiB', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))


In [ ]:
# Cell 4 — tiny GPU smoke (seconds). Stop here if this fails.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
name = 'Qwen/Qwen3-0.6B'
tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, trust_remote_code=True).cuda()
ids = tok('2+2=', return_tensors='pt').to('cuda')
out = model.generate(**ids, max_new_tokens=4, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))
del model
torch.cuda.empty_cache()
print('SMOKE OK')


In [ ]:
# Cell 5 — pipeline_prep ONCE (baseline eval; thinking disabled in code)
import os, sys, json, time
from pathlib import Path
TASK1 = Path('/content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA3/task1')
ART = TASK1 / 'results_colab'
os.chdir(TASK1)
sys.path.insert(0, str(TASK1))
os.environ['ATML_ARTIFACTS_ROOT'] = str(ART)
os.environ['HF_HOME'] = '/content/hf_home'
os.environ['HF_HUB_CACHE'] = '/content/hf_home'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
t0 = time.time()
rc = os.system(f"{sys.executable} -m src.run_stage --stage pipeline_prep --artifacts-root {ART}")
print('PREP_RC', rc, 'elapsed_min', round((time.time()-t0)/60, 2))
print((ART / 'stage_status.json').read_text() if (ART / 'stage_status.json').exists() else 'no stage_status')
assert rc == 0, 'prep failed — do not start post'


In [ ]:
# Cell 6 — stage ckpts then pipeline_post ONCE
import os, sys, shutil, time
from pathlib import Path
TASK1 = Path('/content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA3/task1')
CKPT_SRC = TASK1 / 'colab_data' / 'checkpoints'
ART = TASK1 / 'results_colab'
dst = ART / 'checkpoints'
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(CKPT_SRC, dst)
print('staged', sorted(p.name for p in dst.iterdir()))
os.chdir(TASK1)
sys.path.insert(0, str(TASK1))
os.environ['ATML_ARTIFACTS_ROOT'] = str(ART)
os.environ['HF_HOME'] = '/content/hf_home'
os.environ['HF_HUB_CACHE'] = '/content/hf_home'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
t0 = time.time()
rc = os.system(f"{sys.executable} -m src.run_stage --stage pipeline_post --artifacts-root {ART}")
print('POST_RC', rc, 'elapsed_min', round((time.time()-t0)/60, 2))
print((ART / 'stage_status.json').read_text() if (ART / 'stage_status.json').exists() else 'no stage_status')
assert rc == 0, 'post failed'
